# 🏥 Uncertainty-Aware Medical VQA — Complete Pipeline

**All-in-one notebook:** Baseline → Fine-Tuning → Uncertainty → Abstention → Safety Analysis

### Skip Flags
Each phase can be **skipped** by setting its flag to `True` in Cell 3 (Config).
- Skipped phases load saved results from disk instead of re-running.

| Phase | What It Does | Time (T4) | Skip Flag |
|-------|-------------|-----------|----------|
| **1. Baseline** | Zero-shot BLIP-2 inference | ~10 min | `SKIP_BASELINE` |
| **2. Fine-Tuning** | LoRA training on stratified subset | ~30 min | `SKIP_TRAINING` |
| **3. Uncertainty** | Entropy + MC Dropout + Abstention | ~30 min | `SKIP_UNCERTAINTY` |
| **4. Comparison** | Final tables + safety plots | ~2 min | Always runs |

**Requirements:** T4 GPU (`Runtime > Change runtime type > T4 GPU`)

---
## Cell 1: Install Dependencies

In [ ]:
%pip install -q transformers accelerate peft bitsandbytes datasets Pillow tqdm pandas scikit-learn nltk bert-score matplotlib

## Cell 2: Mount Google Drive

Run this **first** so Drive is available before paths are defined.
- Set `USE_DRIVE = True` to persist all data/checkpoints/results to your Drive.
- Set `USE_DRIVE = False` to use Colab's ephemeral `/content/` storage (faster, lost on disconnect).

In [ ]:
USE_DRIVE = True  # Set False to skip Drive and use /content/ instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Drive mount — using ephemeral Colab storage.")

## Cell 3: Configuration & Skip Flags

In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# ============================================================
# SKIP FLAGS — Set True to skip a phase and load saved results
# ============================================================
SKIP_BASELINE    = False   # Skip Phase 1: load baseline_summary.json
SKIP_TRAINING    = False   # Skip Phase 2: load LoRA checkpoint
SKIP_UNCERTAINTY = False   # Skip Phase 3: load uncertainty_summary.json

# ============================================================
# PROJECT PATHS — defined AFTER drive is mounted
# ============================================================
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"

PROJECT_DIR = DRIVE_PROJECT_DIR if USE_DRIVE else "/content/kvasir-vqa"
DATA_DIR    = f"{PROJECT_DIR}/data"
IMAGE_DIR   = f"{DATA_DIR}/images"
RESULTS_DIR = f"{PROJECT_DIR}/results"
PRED_DIR    = f"{RESULTS_DIR}/predictions"
UNC_DIR     = f"{RESULTS_DIR}/uncertainty"
CKPT_DIR    = f"{PROJECT_DIR}/checkpoints"

for d in [DATA_DIR, IMAGE_DIR, PRED_DIR, UNC_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# MODEL & TRAINING CONFIG
# ============================================================
MODEL_NAME         = "Salesforce/blip2-opt-2.7b"
SEED               = 42
MAX_NEW_TOKENS     = 64
EVAL_SAMPLES       = 50

# Fine-tuning
TRAIN_SUBSET_SIZE  = 2000
EPOCHS             = 3
BATCH_SIZE         = 4
GRAD_ACCUM_STEPS   = 4
LEARNING_RATE      = 2e-4
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1

# LoRA
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Uncertainty
MC_DROPOUT_PASSES  = 5
TARGET_COVERAGE    = 0.80

print("Configuration ready.")
print(f"  Project dir: {PROJECT_DIR}")
print(f"  Model:       {MODEL_NAME}")
print(f"  Train:       {TRAIN_SUBSET_SIZE} samples, {EPOCHS} epochs")
print(f"  Eval:        {EVAL_SAMPLES} samples")
print(f"  LoRA:        r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Uncertainty: {MC_DROPOUT_PASSES} MC passes, {TARGET_COVERAGE*100:.0f}% target coverage")
print(f"  Skipping:    Baseline={SKIP_BASELINE}, Training={SKIP_TRAINING}, Uncertainty={SKIP_UNCERTAINTY}")

## Cell 4: Load Dataset

In [ ]:
train_csv = Path(DATA_DIR) / "kvasir_vqa_x1_train.csv"
test_csv  = Path(DATA_DIR) / "kvasir_vqa_x1_test.csv"

if train_csv.exists() and test_csv.exists():
    print(f"Data found at {DATA_DIR}")
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)
else:
    print("Downloading dataset from HuggingFace...")
    from datasets import load_dataset
    from tqdm.auto import tqdm

    ds = load_dataset("SimulaMet/Kvasir-VQA-x1")
    train_df = ds['train'].to_pandas()
    test_df  = ds['test'].to_pandas()
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    print("QA pairs saved.")

    print("Downloading images...")
    img_ds = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    for sample in tqdm(img_ds, desc="Saving images"):
        img_id = sample.get('img_id', sample.get('imgId', ''))
        img = sample.get('img', sample.get('image', None))
        if img is not None and img_id:
            save_path = Path(IMAGE_DIR) / f"{img_id}.jpg"
            if not save_path.exists():
                img.save(str(save_path))

n_images = len(list(Path(IMAGE_DIR).glob('*.jpg')))
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Images: {n_images:,}")

# Create eval subset — same seed for all phases (apples-to-apples comparison)
np.random.seed(SEED)
eval_subset = test_df.sample(n=min(EVAL_SAMPLES, len(test_df)), random_state=SEED)
print(f"Eval subset: {len(eval_subset)} samples (fixed seed={SEED})")

## Cell 5: All Metric & Utility Functions

Shared across all phases — define once, use everywhere.

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def compute_exact_match(pred, gt):
    return normalize_text(pred) == normalize_text(gt)

def compute_word_f1(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok and not g_tok: return 1.0
    if not p_tok or not g_tok: return 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    if common == 0: return 0.0
    prec = common / len(p_tok)
    rec  = common / len(g_tok)
    return 2 * prec * rec / (prec + rec)

def compute_word_precision(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok: return 1.0 if not g_tok else 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    return common / len(p_tok)

def compute_word_recall(pred, gt):
    """Critical in medical — missing a finding is worse than a false alarm."""
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return 1.0
    if not p_tok: return 0.0
    common = sum((Counter(p_tok) & Counter(g_tok)).values())
    return common / len(g_tok)

def compute_bleu_scores(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return {f'bleu_{n}': (1.0 if not p_tok else 0.0) for n in range(1,5)}
    if not p_tok: return {f'bleu_{n}': 0.0 for n in range(1,5)}
    smooth = SmoothingFunction().method1
    return {f'bleu_{n}': sentence_bleu([g_tok], p_tok,
             weights=tuple([1.0/n]*n+[0.0]*(4-n)), smoothing_function=smooth)
            for n in range(1, 5)}

def compute_rouge_l(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not p_tok and not g_tok: return 1.0
    if not p_tok or not g_tok: return 0.0
    m, n = len(g_tok), len(p_tok)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if g_tok[i-1]==p_tok[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    if lcs == 0: return 0.0
    return 2*(lcs/n)*(lcs/m)/((lcs/n)+(lcs/m))

def compute_meteor(pred, gt):
    p_tok = normalize_text(pred).split()
    g_tok = normalize_text(gt).split()
    if not g_tok: return 1.0 if not p_tok else 0.0
    if not p_tok: return 0.0
    return nltk_meteor([g_tok], p_tok)

def evaluate_predictions(predictions, ground_truths, complexities, label="Model"):
    em_l, f1_l, prec_l, rec_l = [], [], [], []
    bl1_l, bl2_l, bl3_l, bl4_l, rl_l, met_l = [], [], [], [], [], []
    per_comp = {}

    for pred, gt, comp in zip(predictions, ground_truths, complexities):
        em  = int(compute_exact_match(pred, gt))
        f1  = compute_word_f1(pred, gt)
        p   = compute_word_precision(pred, gt)
        r   = compute_word_recall(pred, gt)
        bl  = compute_bleu_scores(pred, gt)
        rl  = compute_rouge_l(pred, gt)
        met = compute_meteor(pred, gt)

        em_l.append(em); f1_l.append(f1); prec_l.append(p); rec_l.append(r)
        bl1_l.append(bl['bleu_1']); bl2_l.append(bl['bleu_2'])
        bl3_l.append(bl['bleu_3']); bl4_l.append(bl['bleu_4'])
        rl_l.append(rl); met_l.append(met)

        key = f"level_{comp}"
        if key not in per_comp:
            per_comp[key] = {'em':[], 'f1':[], 'rec':[], 'bl1':[], 'bl4':[], 'rl':[], 'met':[]}
        per_comp[key]['em'].append(em); per_comp[key]['f1'].append(f1)
        per_comp[key]['rec'].append(r); per_comp[key]['bl1'].append(bl['bleu_1'])
        per_comp[key]['bl4'].append(bl['bleu_4']); per_comp[key]['rl'].append(rl)
        per_comp[key]['met'].append(met)

    try:
        from bert_score import score as bert_score_fn
        _, _, bs_F1 = bert_score_fn(predictions, ground_truths, lang="en",
                                     verbose=False, rescale_with_baseline=True)
        bertscore = bs_F1.numpy().tolist()
    except Exception:
        bertscore = [0.0] * len(predictions)

    return {
        'label': label, 'n': len(predictions),
        'em': em_l, 'f1': f1_l, 'prec': prec_l, 'rec': rec_l,
        'bl1': bl1_l, 'bl2': bl2_l, 'bl3': bl3_l, 'bl4': bl4_l,
        'rl': rl_l, 'met': met_l, 'bertscore': bertscore,
        'per_complexity': per_comp,
        'avg': {
            'exact_match': np.mean(em_l)*100,   'word_f1': np.mean(f1_l)*100,
            'word_precision': np.mean(prec_l)*100, 'word_recall': np.mean(rec_l)*100,
            'bleu_1': np.mean(bl1_l)*100,  'bleu_2': np.mean(bl2_l)*100,
            'bleu_3': np.mean(bl3_l)*100,  'bleu_4': np.mean(bl4_l)*100,
            'rouge_l': np.mean(rl_l)*100,  'meteor': np.mean(met_l)*100,
            'bertscore_f1': np.mean(bertscore)*100,
        },
    }

def print_results(results):
    a = results['avg']
    print(f"  Samples:        {results['n']}")
    print(f"  Exact Match:    {a['exact_match']:.1f}%")
    print(f"  Word F1:        {a['word_f1']:.1f}%  |  Precision: {a['word_precision']:.1f}%  |  Recall: {a['word_recall']:.1f}%")
    print(f"  BLEU-1/4:       {a['bleu_1']:.1f}% / {a['bleu_4']:.1f}%")
    print(f"  ROUGE-L:        {a['rouge_l']:.1f}%")
    print(f"  METEOR:         {a['meteor']:.1f}%")
    print(f"  BERTScore F1:   {a['bertscore_f1']:.1f}%")
    pc = results['per_complexity']
    if pc:
        print(f"  Per-Complexity:")
        for key in sorted(pc.keys()):
            s = pc[key]
            print(f"    {key}: F1={np.mean(s['f1'])*100:.1f}%, BLEU-1={np.mean(s['bl1'])*100:.1f}%, ROUGE-L={np.mean(s['rl'])*100:.1f}% [n={len(s['em'])}]")

def run_inference(model, processor, eval_df, image_dir, max_tokens=64):
    preds, gts, comps, qs, ids = [], [], [], [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Inference"):
        img_path = Path(image_dir) / f"{row['img_id']}.jpg"
        if not img_path.exists(): continue
        image    = Image.open(img_path).convert('RGB')
        question = str(row['question'])
        gt       = str(row['answer'])
        prompt   = f"Question: {question} Answer:"
        inputs   = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, num_beams=3)
        prompt_len = inputs['input_ids'].shape[1]
        new_tokens = generated[0][prompt_len:]
        prediction = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not prediction:
            prediction = processor.tokenizer.decode(generated[0], skip_special_tokens=True).strip()
            if "Answer:" in prediction:
                prediction = prediction.split("Answer:")[-1].strip()
        preds.append(prediction); gts.append(gt)
        comps.append(int(row.get('complexity', 1))); qs.append(question)
        ids.append(row['img_id'])
    return preds, gts, comps, qs, ids

print("All metric/utility functions loaded. ✓")

---
# ═══════════════════════════════════════════
# PHASE 1 — Baseline (Zero-Shot BLIP-2)
# ═══════════════════════════════════════════

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

if SKIP_BASELINE:
    print("⏩ SKIPPING Phase 1 — Loading saved baseline results...")
    with open(f"{PRED_DIR}/baseline_summary.json") as f:
        baseline_saved = json.load(f)
    baseline_results = {
        'avg': baseline_saved['metrics'],
        'label': 'Baseline',
        'n': baseline_saved['num_eval_samples'],
        'per_complexity': baseline_saved.get('per_complexity', {}),
        'f1': [], 'bl1': [], 'rl': [], 'met': [], 'bertscore': [],
    }
    print(f"  Loaded: Word F1 = {baseline_saved['metrics']['word_f1']:.1f}%")
else:
    print("🔵 PHASE 1: Baseline Inference (Zero-Shot BLIP-2)")
    print("="*60)
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    processor = Blip2Processor.from_pretrained(MODEL_NAME)
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    print(f"Loading {MODEL_NAME} in 8-bit...")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.float16,
    )
    model.eval()
    print("Model loaded. Running zero-shot inference...")

    bl_preds, bl_gts, bl_comps, bl_qs, bl_ids = run_inference(
        model, processor, eval_subset, IMAGE_DIR, MAX_NEW_TOKENS
    )

    print("\nComputing metrics...")
    baseline_results = evaluate_predictions(bl_preds, bl_gts, bl_comps, label="Baseline (Zero-Shot)")

    print(f"\n{'='*60}")
    print(f"  BASELINE RESULTS")
    print(f"{'='*60}")
    print_results(baseline_results)
    print(f"{'='*60}")

    # Save
    baseline_summary = {
        'model': MODEL_NAME, 'method': 'zero-shot',
        'num_eval_samples': baseline_results['n'],
        'metrics': {k: round(v, 2) for k, v in baseline_results['avg'].items()},
        'per_complexity': {
            k: {mk: round(np.mean(mv)*100, 2) for mk, mv in v.items()}
            for k, v in baseline_results['per_complexity'].items()
        },
    }
    with open(f"{PRED_DIR}/baseline_summary.json", 'w') as f:
        json.dump(baseline_summary, f, indent=2)

    pd.DataFrame({
        'img_id': bl_ids, 'question': bl_qs, 'ground_truth': bl_gts,
        'prediction': bl_preds, 'complexity': bl_comps,
        'word_f1': [round(x,3) for x in baseline_results['f1']],
        'bleu_1':  [round(x,3) for x in baseline_results['bl1']],
        'rouge_l': [round(x,3) for x in baseline_results['rl']],
        'meteor':  [round(x,3) for x in baseline_results['met']],
        'bertscore_f1': [round(x,3) for x in baseline_results['bertscore']],
    }).to_csv(f"{PRED_DIR}/baseline_predictions.csv", index=False)

    print(f"\n✓ Results saved to {PRED_DIR}/baseline_*.")

---
# ═══════════════════════════════════════════
# PHASE 2 — LoRA Fine-Tuning
# ═══════════════════════════════════════════

### Cell 7: Prepare LoRA Model

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, TaskType

def _load_base_model():
    """Load BLIP-2 base model if not already in memory."""
    global processor, model
    if 'processor' not in globals() or processor is None:
        processor = Blip2Processor.from_pretrained(MODEL_NAME)
        if processor.tokenizer.pad_token is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token
    if 'model' not in globals() or model is None:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = Blip2ForConditionalGeneration.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )

if SKIP_TRAINING:
    print("⏩ SKIPPING Phase 2 — Loading LoRA checkpoint...")
    _load_base_model()

    lora_path = None
    for name in ["best_lora", "final_lora"]:
        p = f"{CKPT_DIR}/{name}"
        if os.path.exists(p):
            lora_path = p; break

    if lora_path:
        model = PeftModel.from_pretrained(model, lora_path)
        print(f"  LoRA loaded from: {lora_path}")
    else:
        print("  ⚠️ No LoRA checkpoint found — using base model weights.")

    model.eval()

    ft_path = f"{PRED_DIR}/finetuned_summary.json"
    if os.path.exists(ft_path):
        with open(ft_path) as f:
            ft_saved = json.load(f)
        finetuned_results = {
            'avg': ft_saved.get('metrics', {}),
            'label': 'Fine-Tuned', 'n': ft_saved.get('num_eval_samples', 0),
            'per_complexity': ft_saved.get('per_complexity', {}),
            'f1': [], 'bl1': [], 'rl': [], 'met': [], 'bertscore': [],
        }
        print(f"  Loaded saved FT results: Word F1 = {finetuned_results['avg'].get('word_f1', 'N/A')}%")
    else:
        finetuned_results = None
        print("  No saved FT results — will evaluate after loading checkpoint.")

else:
    print("🟢 PHASE 2: LoRA Fine-Tuning")
    print("="*60)
    _load_base_model()

    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
    finetuned_results = None

### Cell 8: Training Loop

In [ ]:
if not SKIP_TRAINING:
    from torch.utils.data import Dataset, DataLoader
    from transformers import get_cosine_schedule_with_warmup

    # Stratified training subset
    np.random.seed(SEED)
    if 'complexity' in train_df.columns:
        levels = sorted(train_df['complexity'].unique())
        per_level = TRAIN_SUBSET_SIZE // len(levels)
        train_subset = pd.concat([
            train_df[train_df['complexity'] == lvl].sample(
                n=min(per_level, len(train_df[train_df['complexity'] == lvl])), random_state=SEED)
            for lvl in levels
        ]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    else:
        train_subset = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED)
    print(f"Training subset: {len(train_subset):,} samples")

    class VQAFineTuneDataset(Dataset):
        def __init__(self, df, image_dir, processor, max_length=128):
            self.df = df.reset_index(drop=True)
            self.image_dir = Path(image_dir)
            self.processor = processor
            self.max_length = max_length
            valid = self.df['img_id'].apply(lambda x: (self.image_dir / f"{x}.jpg").exists())
            self.df = self.df[valid].reset_index(drop=True)
            print(f"  Dataset: {len(self.df)} samples with images")

        def __len__(self): return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image    = Image.open(self.image_dir / f"{row['img_id']}.jpg").convert('RGB')
            question = str(row['question'])
            answer   = str(row['answer'])
            prompt   = f"Question: {question} Answer:"
            full     = f"Question: {question} Answer: {answer}"

            prompt_len = self.processor.tokenizer(
                prompt, add_special_tokens=False, return_tensors="pt").input_ids.shape[1]

            enc = self.processor(images=image, text=full, return_tensors="pt",
                                  padding="max_length", truncation=True, max_length=self.max_length)
            input_ids      = enc.input_ids.squeeze()
            attention_mask = enc.attention_mask.squeeze()
            pixel_values   = enc.pixel_values.squeeze()

            labels = input_ids.clone()
            labels[:prompt_len] = -100
            labels[labels == self.processor.tokenizer.pad_token_id] = -100

            return {'pixel_values': pixel_values, 'input_ids': input_ids,
                    'attention_mask': attention_mask, 'labels': labels}

    train_dataset = VQAFineTuneDataset(train_subset, IMAGE_DIR, processor)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    optimizer    = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                      lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps  = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    print(f"Effective batch: {BATCH_SIZE}x{GRAD_ACCUM_STEPS}={BATCH_SIZE*GRAD_ACCUM_STEPS} | Steps: {total_steps}")

    model.train()
    best_loss  = float('inf')
    start_time = time.time()
    training_log = []

    for epoch in range(EPOCHS):
        epoch_loss, step_count = 0.0, 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            pv = batch['pixel_values'].to(model.device, dtype=torch.float16)
            ii = batch['input_ids'].to(model.device)
            am = batch['attention_mask'].to(model.device)
            lb = batch['labels'].to(model.device)

            out  = model(pixel_values=pv, input_ids=ii, attention_mask=am, labels=lb)
            loss = out.loss / GRAD_ACCUM_STEPS
            loss.backward()
            epoch_loss += out.loss.item()
            step_count += 1

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            pbar.set_postfix({'loss': f'{epoch_loss/step_count:.4f}',
                              'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        avg_loss = epoch_loss / step_count
        elapsed  = time.time() - start_time
        training_log.append({'epoch': epoch+1, 'avg_loss': avg_loss, 'elapsed_min': elapsed/60})
        print(f"  Epoch {epoch+1}: Loss={avg_loss:.4f}, Time={elapsed/60:.1f}min")

        if avg_loss < best_loss:
            best_loss = avg_loss
            model.save_pretrained(f"{CKPT_DIR}/best_lora")
            print(f"  ✓ Best checkpoint saved (loss={best_loss:.4f})")

    model.save_pretrained(f"{CKPT_DIR}/final_lora")
    with open(f"{RESULTS_DIR}/training_log.json", 'w') as f:
        json.dump(training_log, f, indent=2)
    total_time = time.time() - start_time
    print(f"\n✓ Done! {total_time/60:.1f}min | Best loss: {best_loss:.4f}")

else:
    print("⏩ Training skipped.")

### Cell 9: Evaluate Fine-Tuned Model

In [ ]:
if finetuned_results is None:
    print("Evaluating fine-tuned model on eval subset...")
    model.eval()
    ft_preds, ft_gts, ft_comps, ft_qs, ft_ids = run_inference(
        model, processor, eval_subset, IMAGE_DIR, MAX_NEW_TOKENS)

    finetuned_results = evaluate_predictions(ft_preds, ft_gts, ft_comps, label="Fine-Tuned (LoRA)")

    print(f"\n{'='*60}")
    print(f"  FINE-TUNED RESULTS (LoRA BLIP-2)")
    print(f"{'='*60}")
    print_results(finetuned_results)
    print(f"{'='*60}")

    ft_summary = {
        'model': MODEL_NAME, 'method': 'LoRA fine-tuned',
        'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
        'training_samples': TRAIN_SUBSET_SIZE, 'epochs': EPOCHS,
        'num_eval_samples': finetuned_results['n'],
        'metrics': {k: round(v, 2) for k, v in finetuned_results['avg'].items()},
        'per_complexity': {
            k: {mk: round(np.mean(mv)*100, 2) for mk, mv in v.items()}
            for k, v in finetuned_results['per_complexity'].items()
        },
    }
    with open(f"{PRED_DIR}/finetuned_summary.json", 'w') as f:
        json.dump(ft_summary, f, indent=2)

    pd.DataFrame({
        'img_id': ft_ids, 'question': ft_qs, 'ground_truth': ft_gts,
        'prediction': ft_preds, 'complexity': ft_comps,
        'word_f1': [round(x,3) for x in finetuned_results['f1']],
        'bleu_1':  [round(x,3) for x in finetuned_results['bl1']],
        'rouge_l': [round(x,3) for x in finetuned_results['rl']],
        'meteor':  [round(x,3) for x in finetuned_results['met']],
    }).to_csv(f"{PRED_DIR}/finetuned_predictions.csv", index=False)

    print(f"✓ Saved to {PRED_DIR}/finetuned_*.")
else:
    print(f"Fine-tuned results already loaded. Word F1 = {finetuned_results['avg'].get('word_f1', 'N/A')}%")

---
# ═══════════════════════════════════════════
# PHASE 3 — Uncertainty Estimation & Abstention
# ═══════════════════════════════════════════

### Cell 10: Uncertainty Functions

In [ ]:
def get_entropy_and_confidence(model, processor, image, question, max_tokens=64):
    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, num_beams=3,
                              output_scores=True, return_dict_in_generate=True)
    prompt_len = inputs['input_ids'].shape[1]
    gen_ids    = out.sequences[0][prompt_len:]
    prediction = processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    entropies, log_probs = [], []
    for step, score in enumerate(out.scores):
        probs = torch.softmax(score[0], dim=-1)
        log_p = torch.log(probs.clamp(min=1e-10))
        entropies.append(-(probs * log_p).sum().item())
        if step < len(gen_ids):
            log_probs.append(log_p[gen_ids[step]].item())

    entropy_mean = float(np.mean(entropies)) if entropies else 0.0
    confidence   = float(np.exp(np.mean(log_probs))) if log_probs else 0.0
    return prediction, entropy_mean, confidence


def enable_dropout(m):
    for mod in m.modules():
        if isinstance(mod, torch.nn.Dropout): mod.train()

def disable_dropout(m):
    for mod in m.modules():
        if isinstance(mod, torch.nn.Dropout): mod.eval()

def mc_dropout_inference(model, processor, image, question, n_passes=5, max_tokens=64):
    prompt     = f"Question: {question} Answer:"
    inputs     = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)
    prompt_len = inputs['input_ids'].shape[1]

    enable_dropout(model)
    answers = []
    for _ in range(n_passes):
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        answers.append(processor.tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True).strip())
    disable_dropout(model)

    pw     = [compute_word_f1(answers[i], answers[j]) for i in range(len(answers)) for j in range(i+1, len(answers))]
    mc_unc = 1.0 - (np.mean(pw) if pw else 1.0)
    normed = [normalize_text(a) for a in answers]
    pred   = answers[normed.index(Counter(normed).most_common(1)[0][0])]
    return pred, answers, float(mc_unc), len(set(normed))/len(normed)

print("Uncertainty functions loaded. ✓")

### Cell 11: Run Uncertainty-Aware Inference

In [ ]:
if SKIP_UNCERTAINTY:
    print("⏩ SKIPPING Phase 3 — Loading saved uncertainty results...")
    unc_csv = f"{UNC_DIR}/uncertainty_predictions.csv"
    if os.path.exists(unc_csv):
        unc_df      = pd.read_csv(unc_csv)
        unc_results = [unc_df.iloc[i].to_dict() for i in range(len(unc_df))]

        required_keys = {"word_f1", "entropy", "confidence", "mc_uncertainty", "combined_uncertainty"}
        if unc_results and not required_keys.issubset(unc_results[0].keys()):
            missing = required_keys - set(unc_results[0].keys())
            print(f"  ⚠️ Loaded CSV is missing columns: {missing} — re-run Phase 3 without skip.")
            unc_results = []
        print(f"  Loaded {len(unc_results)} samples.")
    else:
        print("  ⚠️ No saved uncertainty CSV found!")
        unc_results = []
else:
    print("🔴 PHASE 3: Uncertainty Estimation")
    print("="*60)
    print(f"  1 greedy pass (entropy + log-prob) + {MC_DROPOUT_PASSES} MC passes per sample")
    print(f"  Total generations: ~{len(eval_subset) * (1 + MC_DROPOUT_PASSES)}\n")

    model.eval()
    unc_results = []
    start = time.time()

    for _, row in tqdm(eval_subset.iterrows(), total=len(eval_subset), desc="Uncertainty eval"):
        img_path = Path(IMAGE_DIR) / f"{row['img_id']}.jpg"
        if not img_path.exists(): continue

        image    = Image.open(img_path).convert('RGB')
        question = str(row['question'])
        gt       = str(row['answer'])
        comp     = int(row.get('complexity', 1))

        pred_g, entropy, confidence = get_entropy_and_confidence(model, processor, image, question, MAX_NEW_TOKENS)
        pred_mc, mc_ans, mc_unc, unique_ratio = mc_dropout_inference(model, processor, image, question, MC_DROPOUT_PASSES, MAX_NEW_TOKENS)

        prediction   = pred_mc
        entropy_norm = min(entropy / 10.0, 1.0)
        combined     = 0.4 * entropy_norm + 0.3 * mc_unc + 0.3 * (1.0 - confidence)

        bl  = compute_bleu_scores(prediction, gt)
        unc_results.append({
            'img_id': row['img_id'], 'question': question, 'ground_truth': gt,
            'prediction': prediction, 'complexity': comp,
            'exact_match': int(compute_exact_match(prediction, gt)),
            'word_f1': compute_word_f1(prediction, gt),
            'bleu_1': bl['bleu_1'], 'bleu_4': bl['bleu_4'],
            'rouge_l': compute_rouge_l(prediction, gt),
            'meteor': compute_meteor(prediction, gt),
            'entropy': entropy, 'confidence': confidence,
            'mc_uncertainty': mc_unc, 'mc_unique_ratio': unique_ratio,
            'combined_uncertainty': combined,
        })

    elapsed = time.time() - start
    print(f"\n✓ Done! {elapsed/60:.1f}min | {elapsed/max(1,len(unc_results)):.1f}s per sample")

### Cell 12: Abstention Threshold Tuning & Safety Metrics

In [ ]:
if len(unc_results) == 0:
    print("⚠️ No uncertainty results. Skipping safety analysis.")
else:
    f1_scores     = [r['word_f1'] for r in unc_results]
    entropies     = [r['entropy'] for r in unc_results]
    confidences   = [r['confidence'] for r in unc_results]
    mc_uncs       = [r['mc_uncertainty'] for r in unc_results]
    combined_uncs = [r['combined_uncertainty'] for r in unc_results]
    comps         = [r['complexity'] for r in unc_results]

    print("Correlation (uncertainty vs F1, negative is good):")
    for name, vals in [('Entropy', entropies), ('MC Dropout', mc_uncs),
                       ('1-Confidence', [1-c for c in confidences]), ('Combined', combined_uncs)]:
        r = np.corrcoef(vals, f1_scores)[0,1]
        print(f"  {name:<15} r = {r:+.3f}")

    # Threshold tuning
    unc_arr = np.array(combined_uncs)
    f1_arr  = np.array(f1_scores)
    best_t, best_acc, best_cov = unc_arr.max(), float(f1_arr.mean()), 1.0  # fallback: answer everything
    for t in np.linspace(unc_arr.min(), unc_arr.max(), 100):
        mask = unc_arr <= t
        cov  = mask.sum() / len(unc_arr)
        sel  = f1_arr[mask].mean() if mask.sum() > 0 else 0
        if cov >= TARGET_COVERAGE and sel > best_acc:
            best_t, best_acc, best_cov = t, sel, cov

    answered  = [i for i, u in enumerate(combined_uncs) if u <= best_t]
    abstained = [i for i, u in enumerate(combined_uncs) if u > best_t]

    print(f"\n{'='*60}")
    print(f"  ABSTENTION RESULTS")
    print(f"{'='*60}")
    print(f"  Threshold τ:    {best_t:.4f}")
    print(f"  Coverage:       {best_cov*100:.1f}% ({len(answered)}/{len(unc_results)} answered)")
    print(f"  Selective F1:   {best_acc*100:.1f}%")
    print(f"  Overall F1:     {np.mean(f1_scores)*100:.1f}%")
    print(f"  Gain:           +{(best_acc - np.mean(f1_scores))*100:.1f}%")
    print(f"  Abstained:      {len(abstained)} samples")
    print(f"\n  By Complexity:")
    for lvl in sorted(set(comps)):
        idx = [i for i, c in enumerate(comps) if c == lvl]
        ab  = sum(1 for i in idx if combined_uncs[i] > best_t)
        print(f"    Level {lvl}: {ab}/{len(idx)} abstained ({ab/len(idx)*100:.0f}%)")

    # AUROC
    binary_correct = (f1_arr >= 0.5).astype(int)
    inc_idx = np.where(binary_correct == 0)[0]
    cor_idx = np.where(binary_correct == 1)[0]
    if len(inc_idx) > 0 and len(cor_idx) > 0:
        concordant = sum(1 if unc_arr[i] > unc_arr[j] else 0.5 if unc_arr[i] == unc_arr[j] else 0
                         for i in inc_idx for j in cor_idx)
        auroc = concordant / (len(inc_idx) * len(cor_idx))
    else:
        auroc = 0.5

    # Risk-Coverage
    sorted_idx = np.argsort(unc_arr)
    sorted_f1  = f1_arr[sorted_idx]
    coverages  = [(n+1)/len(sorted_f1) for n in range(len(sorted_f1))]
    sel_accs   = [sorted_f1[:n+1].mean() for n in range(len(sorted_f1))]
    auc_risk   = float(np.trapz([1-a for a in sel_accs], coverages))

    # ECE
    conf_arr = np.array(confidences)
    bins = np.linspace(0, 1, 11)
    ece = 0; ece_data = []
    for i in range(10):
        mask = (conf_arr >= bins[i]) & (conf_arr < bins[i+1]) if i < 9 else (conf_arr >= bins[i]) & (conf_arr <= bins[i+1])
        nb = mask.sum()
        if nb == 0: ece_data.append((0,0,0)); continue
        ac, ag = conf_arr[mask].mean(), f1_arr[mask].mean()
        ece += (nb/len(conf_arr)) * abs(ag - ac)
        ece_data.append((float(ac), float(ag), int(nb)))

    # Selective accuracy table
    sel_acc_table = {t: float(sorted_f1[:max(1, int(t*len(sorted_f1)))].mean())
                     for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}

    print(f"\n{'='*60}")
    print(f"  SAFETY METRICS")
    print(f"{'='*60}")
    print(f"  AUROC:    {auroc:.3f}  {'(good → uncertainty is informative)' if auroc > 0.6 else '(fair)' if auroc > 0.5 else '(poor)'}")
    print(f"  AUC-Risk: {auc_risk:.3f}  (lower = safer)")
    print(f"  ECE:      {ece:.3f}  (lower = better calibrated)")
    print(f"\n  Selective Word F1 at coverage levels:")
    for cov_k, acc_v in sel_acc_table.items():
        mark = " ← target" if abs(cov_k - TARGET_COVERAGE) < 0.01 else ""
        print(f"    {cov_k*100:>5.0f}% → {acc_v*100:.1f}%{mark}")
    print(f"{'='*60}")

### Cell 13: Safety Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

if len(unc_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    ax = axes[0, 0]
    ax.plot(coverages, [a*100 for a in sel_accs], 'b-', lw=2, label='Selective F1')
    ax.axhline(y=np.mean(f1_scores)*100, color='r', ls='--', alpha=.7,
               label=f'Overall F1 ({np.mean(f1_scores)*100:.1f}%)')
    ax.axvline(x=TARGET_COVERAGE, color='g', ls=':', alpha=.7,
               label=f'Target ({TARGET_COVERAGE*100:.0f}% coverage)')
    ax.set_xlabel('Coverage'); ax.set_ylabel('Selective Word F1 (%)')
    ax.set_title(f'Risk-Coverage Curve'); ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    ax = axes[0, 1]
    colors = ['#2ecc71' if f >= 0.5 else '#e74c3c' for f in f1_scores]
    ax.scatter(combined_uncs, [f*100 for f in f1_scores], c=colors, alpha=.7, s=50, edgecolors='w', lw=.5)
    ax.axvline(x=best_t, color='orange', ls='--', lw=2, label=f'τ={best_t:.3f} (abstain threshold)')
    ax.set_xlabel('Combined Uncertainty'); ax.set_ylabel('Word F1 (%)')
    ax.set_title('Uncertainty vs Answer Quality'); ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    ax = axes[1, 0]
    bc = [d[0] for d in ece_data if d[2] > 0]
    ba = [d[1] for d in ece_data if d[2] > 0]
    ax.bar(bc, ba, width=.08, alpha=.7, color='#3498db', label='Actual F1 in bin')
    ax.plot([0,1], [0,1], 'r--', alpha=.7, label='Perfect calibration')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Actual Accuracy (F1)')
    ax.set_title(f'Reliability Diagram (ECE={ece:.3f})')
    ax.legend(fontsize=9); ax.grid(True, alpha=.3)
    ax.set_xlim(-.05,1.05); ax.set_ylim(-.05,1.05)

    ax = axes[1, 1]
    for lvl in sorted(set(comps)):
        lvl_u = [combined_uncs[i] for i, c in enumerate(comps) if c == lvl]
        ax.hist(lvl_u, bins=15, alpha=.5, label=f'Level {lvl} (n={len(lvl_u)})')
    ax.axvline(x=best_t, color='orange', ls='--', lw=2, label=f'τ={best_t:.3f}')
    ax.set_xlabel('Combined Uncertainty'); ax.set_ylabel('Count')
    ax.set_title('Uncertainty Distribution by Complexity')
    ax.legend(fontsize=9); ax.grid(True, alpha=.3)

    plt.suptitle('Uncertainty-Aware Medical VQA — Safety Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{UNC_DIR}/safety_plots.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {UNC_DIR}/safety_plots.png")
else:
    print("No uncertainty data — skipping plots.")

---
# ═══════════════════════════════════════════
# PHASE 4 — Final Comparison & Output
# ═══════════════════════════════════════════

### Cell 14: Final Comparison Table

In [ ]:
print("\n" + "="*80)
print("  FINAL COMPARISON: Baseline vs Fine-Tuned vs Uncertainty-Aware")
print("="*80)

bl_avg = baseline_results.get('avg', {})
ft_avg = finetuned_results.get('avg', {}) if finetuned_results else {}

if len(unc_results) > 0:
    unc_avg = {
        'exact_match': np.mean([r['exact_match'] for r in unc_results])*100,
        'word_f1':     np.mean(f1_scores)*100,
        'bleu_1':      np.mean([r['bleu_1'] for r in unc_results])*100,
        'bleu_4':      np.mean([r['bleu_4'] for r in unc_results])*100,
        'rouge_l':     np.mean([r['rouge_l'] for r in unc_results])*100,
        'meteor':      np.mean([r['meteor'] for r in unc_results])*100,
    }
    sel_f1 = best_acc * 100
else:
    unc_avg = {}; sel_f1 = 0

def val(d, k): return f"{d[k]:.1f}%" if k in d and d[k] is not None else "-"

rows = [
    ('Exact Match', 'exact_match'), ('Word F1', 'word_f1'),
    ('BLEU-1', 'bleu_1'), ('BLEU-4', 'bleu_4'),
    ('ROUGE-L', 'rouge_l'), ('METEOR', 'meteor'), ('BERTScore F1', 'bertscore_f1'),
]
print(f"  {'Metric':<18} {'Baseline':>10} {'Fine-Tuned':>12} {'Unc-Aware':>11} {'Selective@80%':>14}")
print(f"  {'-'*70}")
for name, key in rows:
    s = f"{sel_f1:.1f}%" if key == 'word_f1' and len(unc_results) > 0 else "-"
    print(f"  {name:<18} {val(bl_avg,key):>10} {val(ft_avg,key):>12} {val(unc_avg,key):>11} {s:>14}")

if len(unc_results) > 0:
    print(f"\n  Safety Metrics (Uncertainty-Aware only):")
    print(f"  {'-'*40}")
    print(f"  Coverage:    {best_cov*100:.1f}%  ({len(answered)} answered, {len(abstained)} abstained)")
    print(f"  AUROC:       {auroc:.3f}")
    print(f"  AUC-Risk:    {auc_risk:.3f}")
    print(f"  ECE:         {ece:.3f}")
    print(f"\n  • Selective F1 ({sel_f1:.1f}%) > Overall F1 ({np.mean(f1_scores)*100:.1f}%) \u2014 abstention works")

print("="*80)

### Cell 15: Sample Predictions

In [ ]:
print("\nSample Predictions")
print("="*90)
for i, r in enumerate(unc_results[:15]):
    f1  = r['word_f1']
    unc = r.get('combined_uncertainty', 0)
    if unc > best_t:       tag = "🚫 ABSTAIN"
    elif r['exact_match']: tag = "✓ EXACT"
    elif f1 >= 0.5:        tag = "~ PARTIAL"
    else:                  tag = "✗ WRONG"
    print(f"[{i+1:2}] {tag} | C{r['complexity']} | F1={f1:.2f} | Unc={unc:.3f} | Conf={r.get('confidence',0):.3f}")
    print(f"  Q:    {r['question'][:80]}")
    print(f"  GT:   {r['ground_truth'][:80]}")
    print(f"  Pred: {r['prediction'][:80]}")
    print("-"*90)

### Cell 16: Save All Results & Download

In [ ]:
# Save uncertainty outputs
if len(unc_results) > 0:
    unc_summary = {
        'model': MODEL_NAME, 'method': 'LoRA + uncertainty abstention',
        'mc_dropout_passes': MC_DROPOUT_PASSES,
        'target_coverage': TARGET_COVERAGE,
        'eval_samples': len(unc_results),
        'vqa_metrics': {k: round(v, 2) for k, v in unc_avg.items()},
        'safety_metrics': {'auroc': round(auroc,4), 'auc_risk': round(auc_risk,4), 'ece': round(ece,4)},
        'abstention': {
            'threshold': round(best_t,4), 'coverage': round(best_cov,4),
            'selective_f1': round(sel_f1,2), 'overall_f1': round(np.mean(f1_scores)*100,2),
            'n_answered': len(answered), 'n_abstained': len(abstained),
        },
        'selective_accuracy': {f"{int(k*100)}pct": round(v*100,2) for k,v in sel_acc_table.items()},
    }
    with open(f"{UNC_DIR}/uncertainty_summary.json", 'w') as f:
        json.dump(unc_summary, f, indent=2)

    unc_df = pd.DataFrame(unc_results)
    unc_df['abstained'] = [combined_uncs[i] > best_t for i in range(len(unc_results))]
    unc_df.to_csv(f"{UNC_DIR}/uncertainty_predictions.csv", index=False)
    print("Uncertainty outputs saved.")

# List all files
print("\nAll output files:")
for folder in [PRED_DIR, UNC_DIR, CKPT_DIR]:
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            for fn in files:
                fp = os.path.join(root, fn)
                print(f"  {fp}  ({os.path.getsize(fp)/1024:.1f} KB)")

# Download (only if NOT using Drive — Drive saves automatically)
if not USE_DRIVE:
    try:
        from google.colab import files
        for fp in [
            f"{PRED_DIR}/baseline_summary.json",
            f"{PRED_DIR}/finetuned_summary.json",
            f"{UNC_DIR}/uncertainty_summary.json",
            f"{UNC_DIR}/safety_plots.png",
        ]:
            if os.path.exists(fp): files.download(fp)
    except Exception as e:
        print(f"Download error: {e} — use the Files panel instead.")
else:
    print("\n✓ All results are saved to Google Drive — no download needed.")

print("\n🎉 Pipeline complete!")